In [2]:
import pandas as pd
import numpy as np


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [5]:
import tensorflow as tf

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Dense, Dropout

from tensorflow.keras.callbacks import EarlyStopping

In [6]:
np.random.seed(42)

tf.random.set_seed(42)

In [8]:
print("1. Fetching the real IBM Telco Customer Churn dataset...")

url = "https://raw.githubusercontent.com/carlosfab/dsnp2/master/datasets/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(url)

1. Fetching the real IBM Telco Customer Churn dataset...


In [10]:
print("2. Cleaning and preprocessing real data...")
df = df.drop(columns=["customerID"])

2. Cleaning and preprocessing real data...


In [11]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

In [12]:
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})


In [13]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

In [14]:
X = pd.get_dummies(X, drop_first=True)


In [15]:
print("3. Splitting data into train, validation, and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

3. Splitting data into train, validation, and test sets...


In [16]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

In [17]:
print("4. Scaling features...")
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)


4. Scaling features...


In [18]:
print("5. Building the ANN model...")
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])


5. Building the ANN model...


c:\Users\Dell\Documents\c\tensorflow\.venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [20]:
early_stop = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

In [22]:
print("6. Training the model...")
model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)



6. Training the model...
Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8031 - loss: 0.4207 - val_accuracy: 0.7884 - val_loss: 0.4340
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8062 - loss: 0.4174 - val_accuracy: 0.7920 - val_loss: 0.4342
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8073 - loss: 0.4136 - val_accuracy: 0.7911 - val_loss: 0.4365
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8076 - loss: 0.4050 - val_accuracy: 0.7893 - val_loss: 0.4365
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8118 - loss: 0.4056 - val_accuracy: 0.7929 - val_loss: 0.4360
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8127 - loss: 0.4067 - val_accuracy: 0.7902 - val_loss: 0.4375
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8113 - loss: 0.4031 - val_accuracy: 0.7902 - val_loss: 0.4363
Epoch 8/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8118 

In [24]:
print("\n7. Evaluating the model...")
y_pred_probs = model.predict(X_test)
y_pred = (y_pred_probs.ravel() > 0.5).astype(int)



7. Evaluating the model...
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [25]:
print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("\n--- Accuracy ---")
print(accuracy_score(y_test, y_pred))


--- Confusion Matrix ---
[[928 105]
 [180 194]]

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      1033
           1       0.65      0.52      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407


--- Accuracy ---
0.7974413646055437
